# ЛР-1. Работа с данными: очистка и отбор признаков

Датасет: ID_data_mass_18122012 (скважинные измерения).  
Целевые переменные: `G_total`, `КГФ`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlcore.tabular.workflow import run_tabular_feature_review
from mlcore.tabular.preprocessing import normalize_missing_values

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

In [ ]:
ROOT = Path.cwd()
for p in [ROOT, ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
    if (p / 'mlcore').exists():
        ROOT = p
        break

LAB_DIR = ROOT / 'labs/lr-1'
DATA_PATH = LAB_DIR / 'data/dataset.csv'
ASSETS_DIR = LAB_DIR / 'assets'
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка и первичный осмотр

In [ ]:
df = pd.read_csv(DATA_PATH, sep=',', decimal=',', header=1, skiprows=[2], na_values=['-', ''])
print(f'Shape: {df.shape}')
df.head()

## 2. Очистка данных

- Удаляем столбцы-метаданные (`Unnamed: 0` — номер скважины, `Unnamed: 1` — дата, `КГФ.1` — дубликат).
- Конвертируем строковые столбцы с десятичной запятой в числовые.

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'КГФ.1'])

# Строки с десятичной запятой -> числа
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Shape after cleanup: {df.shape}')
print(f'dtypes: all numeric = {(df.dtypes != "object").all()}')

In [ ]:
# Пропуски
df_norm = normalize_missing_values(df)
missing = df_norm.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_norm) * 100).round(1)
pd.DataFrame({'na_count': missing, 'na_pct': missing_pct}).query('na_count > 0')

## 3. Полный анализ признаков

Используем `run_tabular_feature_review` — единый вызов, который выполняет:
- Удаление строк где оба таргета пусты
- Gain ratio по каждому таргету
- Корреляционную матрицу (абсолютные значения)
- Описательную статистику и квартили
- Сохранение всех графиков

In [ ]:
TARGETS = ['G_total', 'КГФ']

artifacts = run_tabular_feature_review(
    df, target_columns=TARGETS, plots_output_dir=ASSETS_DIR, score_bins=10
)

print(f'Cleaned shape: {artifacts.cleaned_df.shape}')
print(f'Features: {len(artifacts.feature_names)}')

### 3.1. Gain ratio (информативность признаков)

In [ ]:
artifacts.score_mean.to_frame('mean_gain_ratio')

### 3.2. Корреляционная матрица

![heatmap](../assets/abs_correlation_heatmap.png)

In [ ]:
# Пары с корреляцией > 0.95 (мультиколлинеарность)
corr = artifacts.abs_corr_matrix
high_corr = []
for i in range(len(corr)):
    for j in range(i + 1, len(corr)):
        if corr.iloc[i, j] > 0.95:
            high_corr.append((corr.index[i], corr.columns[j], round(corr.iloc[i, j], 3)))

pd.DataFrame(high_corr, columns=['Feature A', 'Feature B', '|r|']).sort_values('|r|', ascending=False)

### 3.3. Распределения признаков

Гистограммы с линиями Q1/Q3 сохранены в `assets/distributions/`.

### 3.4. Описательная статистика

In [ ]:
artifacts.descriptive_stats

## 4. Отбор признаков

Критерии:
- **Gain ratio**: средний gain ratio > 0 (информативность для таргетов)
- **Корреляция**: если два признака коррелируют > 0.95, оставляем более информативный
- **Пропуски**: > 80% пропусков — удаляем
- **Вариативность**: константные или почти константные — удаляем

In [ ]:
scores = artifacts.score_mean
missing_rate = artifacts.cleaned_df[artifacts.feature_names].isna().mean()

# Признаки для удаления
DROP = []
KEEP = []
reasons = {}

for feat in artifacts.feature_names:
    if missing_rate[feat] > 0.8:
        DROP.append(feat)
        reasons[feat] = f'>{80}% пропусков ({missing_rate[feat]:.0%})'
    elif scores.get(feat, 0) == 0.0:
        DROP.append(feat)
        reasons[feat] = 'gain ratio = 0 (не информативен)'
    else:
        KEEP.append(feat)
        reasons[feat] = f'gain ratio = {scores.get(feat, 0):.3f}'

# Проверяем мультиколлинеарность среди KEEP
to_remove = set()
for a, b, r in high_corr:
    if a in KEEP and b in KEEP:
        # Удаляем менее информативный
        worse = b if scores.get(a, 0) >= scores.get(b, 0) else a
        if worse not in to_remove:
            to_remove.add(worse)
            reasons[worse] = f'мультиколлинеарность (|r|={r:.3f})'

for feat in to_remove:
    KEEP.remove(feat)
    DROP.append(feat)

print(f'KEEP: {len(KEEP)} признаков')
print(f'DROP: {len(DROP)} признаков')
print()

decision = pd.DataFrame([
    {'Признак': f, 'Решение': 'Оставить' if f in KEEP else 'Удалить', 'Обоснование': reasons[f]}
    for f in artifacts.feature_names
])
decision

## 5. Выводы

1. Датасет содержит 185 наблюдений, но только 23 имеют непустые значения обоих таргетов.
2. Gain ratio позволил выделить наиболее информативные признаки для предсказания G_total и КГФ.
3. Обнаружены группы сильно коррелирующих признаков — из каждой группы оставлен наиболее информативный.
4. Итоговый набор признаков готов для построения моделей.